In [7]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import numpy as np
import os
from io import BytesIO

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "processeddata"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

In [8]:
def load_blob_csv(blob_name):
    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_name
    )

    data = blob_client.download_blob().readall()
    return pd.read_csv(BytesIO(data))

# EKSEMPEL (tilpass til dine faktiske filer)
df_consumption = load_blob_csv("BREIVE_processed.csv")
df_weather = load_blob_csv("BREIVE_weather.csv")
df_norgespris = load_blob_csv("breive_norgespris.csv")

In [10]:
print(df_consumption.head())
print(df_weather.head())
print(df_norgespris.head())

               metering_point_anonymous            timestamp  value_kwh  \
0  fcf4855f-feca-467b-bad2-ff524b7fdca3  2025-01-17 23:00:00       5.63   
1  fcf4855f-feca-467b-bad2-ff524b7fdca3  2025-01-18 00:00:00       4.21   
2  fcf4855f-feca-467b-bad2-ff524b7fdca3  2025-01-18 01:00:00       3.34   
3  fcf4855f-feca-467b-bad2-ff524b7fdca3  2025-01-18 02:00:00       3.82   
4  fcf4855f-feca-467b-bad2-ff524b7fdca3  2025-01-18 03:00:00       3.04   

  transformer_station  consumption_code  hour  weekday  month  is_weekend  \
0              BREIVE                36    23        4      1       False   
1              BREIVE                36     0        5      1        True   
2              BREIVE                36     1        5      1        True   
3              BREIVE                36     2        5      1        True   
4              BREIVE                36     3        5      1        True   

   is_holiday day_night  norgespris  
0       False     night           0  
1       Fa

In [11]:
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS


def prepare_norgespris_regression_data(
    df_consumption,
    df_weather=None,
    df_norgespris=None,
    station_customers_total=None,
    exclude_consumption_codes=(26,),
):
    df = df_consumption.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["date"] = df["timestamp"].dt.normalize()

    if "norgespris" not in df.columns:
        raise ValueError("df_consumption må inneholde kolonnen 'norgespris'.")

    if station_customers_total is None:
        station_customers_total = df["metering_point_anonymous"].nunique()
    if float(station_customers_total) <= 0:
        raise ValueError("station_customers_total må være et positivt tall.")

    # Fjern kundetyper som ikke kan ha Norgespris (f.eks. 26 = næring).
    if "consumption_code" in df.columns and exclude_consumption_codes:
        df = df[~df["consumption_code"].isin(exclude_consumption_codes)].copy()

    # Periodemarkør 0/1 fra forbruksfilen
    df = df[df["norgespris"].notna()].copy()
    df["norgespris"] = df["norgespris"].astype(int)

    if df_weather is not None:
        weather_df = df_weather.copy()
        weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])
        df = df.merge(weather_df, on="timestamp", how="left")

    if df_norgespris is not None and {"transformer_station", "count_total", "timestamp"}.issubset(df_norgespris.columns):
        station_daily = df_norgespris.copy()
        station_daily["timestamp"] = pd.to_datetime(station_daily["timestamp"])
        station_daily["date"] = station_daily["timestamp"].dt.normalize()

        # Ett daglig nivå per stasjon
        station_daily = (
            station_daily.groupby(["date", "transformer_station"], as_index=False)["count_total"]
            .max()
        )

        # Bruk kjent total kundebase i denne trafostasjonen
        station_daily["station_customers_total"] = float(station_customers_total)

        # Andel med Norgespris = antall med avtale / total kundebase
        station_daily["norgespris_share"] = station_daily["count_total"] / station_daily["station_customers_total"]

        # Slå på stasjonsnivå + dato
        df = df.merge(
            station_daily[["date", "transformer_station", "count_total", "station_customers_total", "norgespris_share"]],
            on=["date", "transformer_station"],
            how="left",
        )

    else:
        df["count_total"] = np.nan
        df["station_customers_total"] = float(station_customers_total)
        df["norgespris_share"] = np.nan

    # df_norgespris finnes kun på dager med norgespris=1. Sett 0 i før-periode.
    df.loc[df["norgespris"] == 0, ["count_total", "norgespris_share"]] = 0.0

    # Robust fallback om det mangler noen match i etter-periode
    df["count_total"] = df["count_total"].fillna(0.0)
    df["norgespris_share"] = df["norgespris_share"].fillna(0.0)

    # Klipp andel til [0, 1] for å unngå ekstreme verdier ved eventuelle datamismatch
    df["norgespris_share"] = df["norgespris_share"].clip(lower=0.0, upper=1.0)

    return df


def fit_best_norgespris_model(df, dependent_variable="value_kwh"):
    required_columns = {
        "metering_point_anonymous",
        "timestamp",
        "transformer_station",
        dependent_variable,
        "norgespris",
        "norgespris_share",
        "air_temperature",
        "wind_speed",
        "precipitation_mm",
        "hour",
        "is_weekend",
        "month",
        "is_holiday",
    }

    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        missing_text = ", ".join(sorted(missing_columns))
        raise ValueError(f"Datasettet mangler nødvendige kolonner: {missing_text}")

    modeling_df = df.copy()
    modeling_df["timestamp"] = pd.to_datetime(modeling_df["timestamp"])
    modeling_df["metering_point_anonymous"] = modeling_df["metering_point_anonymous"].astype(str)
    # Sikrer at indikatorer og tidskomponenter behandles som diskrete numeriske variabler.
    modeling_df["is_weekend"] = modeling_df["is_weekend"].astype(int)
    modeling_df["is_holiday"] = modeling_df["is_holiday"].astype(int)
    modeling_df["hour"] = modeling_df["hour"].astype(int)
    modeling_df["month"] = modeling_df["month"].astype(int)
    modeling_df["norgespris"] = modeling_df["norgespris"].astype(int)
    modeling_df["norgespris_share"] = modeling_df["norgespris_share"].astype(float)
    # Viktig: log1p betyr log(1 + forbruk), ikke vanlig log(forbruk).
    modeling_df["log_value_kwh"] = np.log1p(modeling_df[dependent_variable])

    modeling_df = modeling_df.dropna(
        subset=[
            dependent_variable,
            "norgespris",
            "norgespris_share",
            "air_temperature",
            "wind_speed",
            "precipitation_mm",
            "hour",
            "is_weekend",
            "month",
            "is_holiday",
            "metering_point_anonymous",
        ]
    ).copy()

    panel_df = modeling_df.set_index(["metering_point_anonymous", "timestamp"]).sort_index()

    # Intensitetsmodell: bruker faktisk andel med Norgespris (0-1)
    formula = (
        "log_value_kwh ~ 1 + norgespris_share + air_temperature + wind_speed + precipitation_mm "
        "+ is_weekend + is_holiday + C(hour) + C(month) + EntityEffects"
    )

    model = PanelOLS.from_formula(formula, data=panel_df, drop_absorbed=True)
    results = model.fit(cov_type="robust", low_memory=True, use_lsmr=True)

    beta_share = float(results.params.get("norgespris_share", 0.0))

    # Kontrafaktisk uten Norgespris-andel: hold alt annet likt, fjern kun beta*share-leddet.
    observed_kwh = modeling_df[dependent_variable].astype(float)
    share_effect_log = beta_share * modeling_df["norgespris_share"]
    counterfactual_no_np_kwh = np.expm1(np.log1p(observed_kwh) - share_effect_log)
    attributable_kwh = observed_kwh - counterfactual_no_np_kwh

    post_mask = modeling_df["norgespris"] == 1
    total_observed_post_kwh = float(observed_kwh.loc[post_mask].sum())
    total_attributable_post_kwh = float(attributable_kwh.loc[post_mask].sum())
    attributable_share_of_post_kwh_pct = (
        float(total_attributable_post_kwh / total_observed_post_kwh * 100)
        if total_observed_post_kwh > 0
        else 0.0
    )

    unique_customers = int(modeling_df["metering_point_anonymous"].nunique())
    post_days = int(modeling_df.loc[post_mask, "timestamp"].dt.normalize().nunique()) if post_mask.any() else 0
    attributable_per_customer_post_kwh = (
        float(total_attributable_post_kwh / unique_customers)
        if unique_customers > 0
        else 0.0
    )
    attributable_per_customer_post_day_kwh = (
        float(total_attributable_post_kwh / (unique_customers * post_days))
        if unique_customers > 0 and post_days > 0
        else 0.0
    )

    metrics = {
        "nobs": int(results.nobs),
        "r2_within": float(results.rsquared_within),
        "beta_share": beta_share,
        # Effekt hvis andelen går fra 0% til 100%
        "effect_full_share_pct": float((np.exp(beta_share) - 1) * 100),
        # Mer praktisk tolkning: effekt ved +10 prosentpoeng
        "effect_10pp_pct": float((np.exp(beta_share * 0.10) - 1) * 100),
        "mean_share_post": float(
            modeling_df.loc[post_mask, "norgespris_share"].mean()
        ) if post_mask.any() else 0.0,
        # Total effekt i kWh i etterperioden
        "total_observed_post_kwh": total_observed_post_kwh,
        "total_attributable_post_kwh": total_attributable_post_kwh,
        "attributable_share_of_post_kwh_pct": attributable_share_of_post_kwh_pct,
        "attributable_per_customer_post_kwh": attributable_per_customer_post_kwh,
        "attributable_per_customer_post_day_kwh": attributable_per_customer_post_day_kwh,
    }

    return results, modeling_df, metrics


regression_df = prepare_norgespris_regression_data(
    df_consumption,
    df_weather,
    df_norgespris,
    station_customers_total=None,
    exclude_consumption_codes=(26,),
)
results, analysis_df, metrics = fit_best_norgespris_model(regression_df)

print(f"Observasjoner totalt: {analysis_df.shape[0]:,}")
print(f"Brukt i modell: {metrics['nobs']:,}")
print(f"R2 (within): {metrics['r2_within']:.4f}")
print("Merk: modellen bruker log1p(value_kwh) = log(1 + value_kwh), så effekten gjelder 1 + forbruk.")
print(f"Gjennomsnittlig andel med Norgespris i etterperioden: {metrics['mean_share_post']:.2%}")
print(f"Effekt ved +10 prosentpoeng høyere andel: {metrics['effect_10pp_pct']:.2f}%")
print(f"Effekt ved 0% -> 100% andel: {metrics['effect_full_share_pct']:.2f}%")
print()
print(f"Total observert forbruk i etterperioden: {metrics['total_observed_post_kwh']:,.0f} kWh")
print(f"Estimert total effekt av Norgespris i etterperioden: {metrics['total_attributable_post_kwh']:,.0f} kWh")
print(f"Dette tilsvarer: {metrics['attributable_share_of_post_kwh_pct']:.2f}% av totalforbruket i etterperioden")
print(f"Per kunde i etterperioden: {metrics['attributable_per_customer_post_kwh']:.1f} kWh")
print(f"Per kunde per dag (etterperiode): {metrics['attributable_per_customer_post_day_kwh']:.3f} kWh")
print()
print(results.summary)
print()
print(f"Koeffisient for norgespris_share (log-skala): {metrics['beta_share']:.4f}")
print(f"Standardfeil: {results.std_errors.get('norgespris_share'):.4f}")
print(f"p-verdi: {results.pvalues.get('norgespris_share'):.4g}")

Observasjoner totalt: 5,384,355
Brukt i modell: 5,384,355
R2 (within): 0.1641
Merk: modellen bruker log1p(value_kwh) = log(1 + value_kwh), så effekten gjelder 1 + forbruk.
Gjennomsnittlig andel med Norgespris i etterperioden: 82.96%
Effekt ved +10 prosentpoeng høyere andel: 0.30%
Effekt ved 0% -> 100% andel: 3.02%

Total observert forbruk i etterperioden: 4,278,735 kWh
Estimert total effekt av Norgespris i etterperioden: 166,573 kWh
Dette tilsvarer: 3.89% av totalforbruket i etterperioden
Per kunde i etterperioden: 128.2 kWh
Per kunde per dag (etterperiode): 1.583 kWh

                          PanelOLS Estimation Summary                           
Dep. Variable:          log_value_kwh   R-squared:                        0.1641
Estimator:                   PanelOLS   R-squared (Between):           -2.22e-16
No. Observations:             5384355   R-squared (Within):               0.1641
Date:                Mon, Apr 20 2026   R-squared (Overall):              0.1053
Time:              

In [27]:
print('Nøkkeltall fra regresjonen (andel-modell)')
print(f"Observasjoner i modell: {metrics['nobs']:,}")
print(f"R2 (within): {metrics['r2_within']:.4f}")
print(f"Koeffisient norgespris_share: {results.params.get('norgespris_share'):.6f}")
print(f"Effekt ved +10 prosentpoeng: {(np.exp(results.params.get('norgespris_share') * 0.10) - 1) * 100:.2f}%")
print(f"Effekt ved 0% -> 100%: {(np.exp(results.params.get('norgespris_share')) - 1) * 100:.2f}%")
print(f"Standardfeil: {results.std_errors.get('norgespris_share'):.6f}")
print(f"p-verdi: {results.pvalues.get('norgespris_share'):.6g}")
print()
print(f"Total estimert effekt i etterperioden: {metrics['total_attributable_post_kwh']:,.0f} kWh")
print(f"Andel av totalforbruk i etterperioden: {metrics['attributable_share_of_post_kwh_pct']:.2f}%")
print(f"Per kunde i etterperioden: {metrics['attributable_per_customer_post_kwh']:.1f} kWh")
print(f"Per kunde per dag: {metrics['attributable_per_customer_post_day_kwh']:.3f} kWh")

Nøkkeltall fra regresjonen (andel-modell)
Observasjoner i modell: 5,479,690
R2 (within): 0.1635
Koeffisient norgespris_share: 0.028184
Effekt ved +10 prosentpoeng: 0.28%
Effekt ved 0% -> 100%: 2.86%
Standardfeil: 0.005816
p-verdi: 1.26053e-06

Total estimert effekt i etterperioden: 162,884 kWh
Andel av totalforbruk i etterperioden: 3.66%
Per kunde i etterperioden: 123.2 kWh
Per kunde per dag: 1.521 kWh


In [28]:
mean_log_value_kwh = np.log1p(analysis_df['value_kwh']).mean()
coef_share = results.params.get('norgespris_share')
percent_change_10pp = (np.exp(coef_share * 0.10) - 1) * 100
percent_change_full = (np.exp(coef_share) - 1) * 100

print(f'Gjennomsnittlig log1p(value_kwh): {mean_log_value_kwh:.4f}')
print(f'Policy-effekt ved +10 prosentpoeng andel: {percent_change_10pp:.2f}%')
print(f'Policy-effekt ved 0% -> 100% andel: {percent_change_full:.2f}%')
print()
print(f"Total estimert effekt i etterperioden: {metrics['total_attributable_post_kwh']:,.0f} kWh")
print(f"Per kunde i etterperioden: {metrics['attributable_per_customer_post_kwh']:.1f} kWh")
print(f"Per kunde per dag i etterperioden: {metrics['attributable_per_customer_post_day_kwh']:.3f} kWh")

Gjennomsnittlig log1p(value_kwh): 0.8489
Policy-effekt ved +10 prosentpoeng andel: 0.28%
Policy-effekt ved 0% -> 100% andel: 2.86%

Total estimert effekt i etterperioden: 162,884 kWh
Per kunde i etterperioden: 123.2 kWh
Per kunde per dag i etterperioden: 1.521 kWh
